# 5c Train a Digit Classifier

Train a small CPU classifier, inspect held-out errors, and save a numeric checkpoint without pickle.

**Budget:** 15–25 minutes; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), prepare Python 3.10+ and the packages in the [support guide](README.md), and select that kernel. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


In [ ]:
ALLOW_DOWNLOAD = False
data = load_image_data("mnist", allow_download=ALLOW_DOWNLOAD,
                       cache_dir=OUTPUT / "data", limit_train=3000, limit_test=600, seed=SEED)
print(data.name, data.train_images.shape, data.provenance)


## Fit on training data, evaluate on the held-out test set

This fixed baseline uses logistic regression on normalized pixels. It does not fit preprocessing on test examples. The configuration is fixed before looking at test accuracy. If you want to choose a regularization setting, use the validation workflow in 5e and then evaluate the chosen configuration once on test data.

CPU is the default. For an optional small neural network, install the administrator-approved compatible PyTorch build and call the helper's `train_torch(data, device="cpu")` or explicitly `device="cuda"` after confirming a CUDA GPU. That is a different model, not a fair CPU/GPU speed comparison; 5i benchmarks identical operations.


In [ ]:
model, metrics = train_baseline(data, C=1.0, max_iter=100, seed=SEED)
print(metrics)
from sklearn.metrics import ConfusionMatrixDisplay
prediction = model.predict(data.test_images.reshape(len(data.test_images), -1))
ConfusionMatrixDisplay.from_predictions(data.test_labels, prediction,
    labels=np.arange(len(data.class_names)), display_labels=data.class_names)
plt.title(data.name + ": held-out confusion matrix"); plt.show()


In [ ]:
wrong = np.flatnonzero(prediction != data.test_labels)[:8]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2), squeeze=False)
    for ax, index in zip(axes.flat, wrong):
        ax.imshow(data.test_images[index], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"True {data.test_labels[index]} / predicted {prediction[index]}")
        ax.axis("off")
    plt.show()
else:
    print("No errors in this bounded test subset; this does not establish perfect generalization.")


## Save and reload a safe checkpoint

The helper stores supported logistic-regression parameters as numeric arrays and writes a separate JSON provenance file below, with `allow_pickle=False` on load. It does not load arbitrary serialized Python objects. Keep the dataset name and image shape with the model. A checkpoint saves a model; a notebook saves code and outputs; neither preserves a running kernel.


In [ ]:
from image_lab import save_checkpoint, load_checkpoint
checkpoint = OUTPUT / "digit_baseline.npz"
save_checkpoint(model, checkpoint)
import json
checkpoint.with_suffix(".json").write_text(json.dumps({"dataset": data.name,
    "image_shape": list(data.train_images.shape[1:]), "provenance": data.provenance}, indent=2))
restored = load_checkpoint(checkpoint)
np.testing.assert_array_equal(prediction,
    restored.predict(data.test_images.reshape(len(data.test_images), -1)))
print("Restored predictions match:", checkpoint.name)


## Checkpoint, exercise, and cleanup

**Checkpoint:** The classifier reports held-out accuracy, shows class-specific errors, and reproduces predictions from its numeric checkpoint.

**Try it:** Choose an error and explain a plausible ambiguity. Do not keep tweaking the model on these test errors and continue calling the result an untouched test score.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5d: Explore EMNIST](05d_Explore_EMNIST.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
